# 🌊 California Coastal Flood Risk Analysis

## Identifying Properties at Risk of Flooding in California Coastal Cities

This notebook analyzes **FEMA flood zone data** for properties in California's coastal counties using the **Regrid parcel dataset** available in Wherobots. We cross-reference parcels with **Overture Maps building footprints** and enrich the analysis with **historical weather events**.

### Data Sources
| Dataset | Catalog | Description |
|---------|---------|-------------|
| `regrid_parcels` | `wherobots_open_data.partner_samples` | 6.8M+ CA parcel records with FEMA flood zones, property values, structures |
| `buildings_building` | `wherobots_open_data.overture_maps_foundation` | Building footprints from Overture Maps |
| `weather_events` | `wherobots_pro_data.weather` | Historical weather events (rain, storms) |

### FEMA Flood Zone Risk Tiers
| Zone | Risk Level | Description |
|------|-----------|-------------|
| **V / VE** | 🔴 Highest | Coastal high-velocity flood zones (wave action) |
| **A / AE** | 🟠 High | 1% annual chance flood (100-year floodplain) |
| **AO / AH** | 🟡 High | Shallow flooding / ponding areas |
| **A99** | 🟡 Moderate-High | Protected by federal flood control (under construction) |
| **X (0.2%)** | 🟢 Moderate | 0.2% annual chance (500-year floodplain) |
| **X (minimal)** | ⚪ Low | Minimal flood hazard |

### Scope
The Regrid sample covers **5 California counties**: Los Angeles, Orange, San Diego, Riverside, San Bernardino. Three of these (LA, Orange, San Diego) are **coastal**.

## 1. Setup & Initialization

In [1]:
from sedona.spark import *
from pyspark.sql.functions import expr, col, when, sum as spark_sum, count, avg, round as spark_round, lit, desc
import wkls

# Initialize Sedona Context
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

print("✅ SedonaContext initialized successfully")

In [2]:
# Define constants for the analysis
COASTAL_COUNTIES = ['los-angeles', 'orange', 'san-diego']
ALL_COUNTIES = ['los-angeles', 'orange', 'san-diego', 'riverside', 'san-bernardino']

# FEMA high-risk flood zones (Special Flood Hazard Areas)
HIGH_RISK_ZONES = ['A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99']
COASTAL_ZONES = ['V', 'VE']  # Coastal high-velocity zones

# Table references
PARCELS_TABLE = "wherobots_open_data.partner_samples.regrid_parcels"
BUILDINGS_TABLE = "wherobots_open_data.overture_maps_foundation.buildings_building"
WEATHER_TABLE = "wherobots_pro_data.weather.weather_events"

# Get California boundary using wkls
ca_wkt = wkls.us['ca'].wkt()

print("✅ Analysis constants defined")
print(f"   Coastal counties: {COASTAL_COUNTIES}")
print(f"   High-risk FEMA zones: {HIGH_RISK_ZONES}")
print(f"   Coastal FEMA zones: {COASTAL_ZONES}")

## 2. Load & Explore the Regrid Parcel Data

The Regrid dataset contains detailed parcel-level information including FEMA flood zone designations, property valuations, structure details, and parcel geometries. Let's load California parcels and explore the data.

In [3]:
# Load California parcels with relevant columns
ca_parcels = sedona.sql(f"""
    SELECT
        parcelnumb,
        scity,
        city,
        county,
        state2,
        szip5,
        address,
        fema_flood_zone,
        fema_flood_zone_subtype,
        fema_flood_zone_data_date,
        parval,
        landval,
        improvval,
        saleprice,
        saledate,
        yearbuilt,
        numstories,
        numunits,
        usedesc,
        struct,
        structno,
        ll_bldg_footprint_sqft,
        ll_bldg_count,
        ll_gisacre,
        geometry
    FROM {PARCELS_TABLE}
    WHERE state2 = 'CA'
""")

# Cache for reuse
ca_parcels.createOrReplaceTempView("ca_parcels")

total_count = ca_parcels.count()
print(f"✅ Loaded {total_count:,} California parcels")
print(f"\nSchema (key columns):")
ca_parcels.select("scity", "county", "fema_flood_zone", "fema_flood_zone_subtype", "parval", "yearbuilt", "struct").printSchema()

In [4]:
# Explore FEMA flood zone distribution across all CA parcels
print("📊 FEMA Flood Zone Distribution (California)")
print("=" * 70)

sedona.sql("""
    SELECT
        fema_flood_zone,
        fema_flood_zone_subtype,
        COUNT(*) as parcel_count,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM ca_parcels), 2) as pct_of_total
    FROM ca_parcels
    WHERE fema_flood_zone IS NOT NULL AND fema_flood_zone != ''
    GROUP BY fema_flood_zone, fema_flood_zone_subtype
    ORDER BY parcel_count DESC
""").show(25, truncate=False)

## 3. County-Level Flood Risk Overview

Compare flood risk across all available California counties, with special focus on the three coastal counties.

In [5]:
# County-level flood risk summary
print("📊 County-Level Flood Risk Summary")
print("=" * 90)

county_summary = sedona.sql("""
    SELECT
        county,
        CASE WHEN county IN ('los-angeles', 'orange', 'san-diego') THEN '🏖️ Coastal' ELSE '🏔️ Inland' END as type,
        COUNT(*) as total_parcels,
        SUM(CASE WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99') THEN 1 ELSE 0 END) as high_risk_parcels,
        SUM(CASE WHEN fema_flood_zone IN ('VE', 'V') THEN 1 ELSE 0 END) as coastal_v_zone,
        ROUND(
            SUM(CASE WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99') THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) as pct_high_risk,
        ROUND(
            SUM(CASE WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99') THEN parval ELSE 0 END) / 1e9,
            2
        ) as total_value_at_risk_billions
    FROM ca_parcels
    WHERE fema_flood_zone IS NOT NULL AND fema_flood_zone != ''
    GROUP BY county
    ORDER BY high_risk_parcels DESC
""")

county_summary.show(truncate=False)

In [ ]:
# Coastal vs Inland comparison
print("📊 Coastal vs Inland Flood Risk Comparison")
print("=" * 70)

sedona.sql("""
    SELECT
        CASE WHEN county IN ('los-angeles', 'orange', 'san-diego') THEN 'Coastal' ELSE 'Inland' END as region_type,
        COUNT(*) as total_parcels,
        SUM(CASE WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99') THEN 1 ELSE 0 END) as high_risk_parcels,
        SUM(CASE WHEN fema_flood_zone IN ('VE', 'V') THEN 1 ELSE 0 END) as coastal_v_zone,
        ROUND(
            SUM(CASE WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99') THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) as pct_at_risk,
        ROUND(SUM(CASE WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99') THEN parval ELSE 0 END) / 1e9, 2) as value_at_risk_B,
        ROUND(AVG(CASE WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99') THEN parval END) / 1e6, 2) as avg_at_risk_parcel_value_M
    FROM ca_parcels
    WHERE fema_flood_zone IS NOT NULL AND fema_flood_zone != ''
    GROUP BY CASE WHEN county IN ('los-angeles', 'orange', 'san-diego') THEN 'Coastal' ELSE 'Inland' END
    ORDER BY region_type
""").show(truncate=False)

## 4. Coastal City-Level Deep Dive

Drill down into individual cities within the three coastal counties to identify which cities have the most properties at risk and the highest financial exposure.

In [6]:
# Top coastal cities by number of high-risk flood parcels
print("📊 Top 30 Coastal Cities by High-Risk Flood Parcels")
print("=" * 100)

coastal_city_risk = sedona.sql("""
    SELECT
        scity as city_name,
        county,
        COUNT(*) as total_parcels,
        SUM(CASE WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99') THEN 1 ELSE 0 END) as high_risk_parcels,
        SUM(CASE WHEN fema_flood_zone IN ('VE', 'V') THEN 1 ELSE 0 END) as coastal_v_zone,
        ROUND(
            SUM(CASE WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99') THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) as pct_at_risk,
        ROUND(
            AVG(CASE WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99') THEN parval END) / 1e6,
            2
        ) as avg_at_risk_value_M,
        ROUND(
            SUM(CASE WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99') THEN parval ELSE 0 END) / 1e9,
            2
        ) as total_value_at_risk_B
    FROM ca_parcels
    WHERE county IN ('los-angeles', 'orange', 'san-diego')
        AND fema_flood_zone IS NOT NULL AND fema_flood_zone != ''
        AND scity IS NOT NULL AND scity != ''
    GROUP BY scity, county
    HAVING SUM(CASE WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'VE', 'V', 'A99') THEN 1 ELSE 0 END) > 100
    ORDER BY high_risk_parcels DESC
""")

coastal_city_risk.createOrReplaceTempView("coastal_city_risk")
coastal_city_risk.show(30, truncate=False)

## 5. Coastal V-Zone (Highest Risk) Deep Dive

FEMA V and VE zones represent the **highest risk** coastal areas — subject to wave action and storm surge. These properties face the most extreme flooding threat. Let's identify where they concentrate and what's at stake financially.

In [7]:
# Coastal V-Zone analysis — highest risk properties
print("🔴 Coastal V-Zone (VE/V) Properties — Highest Flood Risk")
print("=" * 90)

v_zone_analysis = sedona.sql("""
    SELECT
        scity as city_name,
        county,
        fema_flood_zone,
        COUNT(*) as v_zone_parcels,
        ROUND(AVG(parval) / 1e6, 2) as avg_value_M,
        ROUND(SUM(parval) / 1e9, 2) as total_value_B,
        ROUND(AVG(yearbuilt), 0) as avg_year_built,
        SUM(CASE WHEN yearbuilt < 1980 THEN 1 ELSE 0 END) as pre_1980_parcels,
        ROUND(AVG(ll_gisacre), 2) as avg_acreage
    FROM ca_parcels
    WHERE fema_flood_zone IN ('V', 'VE')
        AND county IN ('los-angeles', 'orange', 'san-diego')
        AND scity IS NOT NULL AND scity != ''
    GROUP BY scity, county, fema_flood_zone
    HAVING COUNT(*) > 10
    ORDER BY v_zone_parcels DESC
""")

v_zone_analysis.show(20, truncate=False)

# Total V-zone summary
print("\n📈 V-Zone Totals Across Coastal Counties:")
sedona.sql("""
    SELECT
        COUNT(*) as total_v_zone_parcels,
        ROUND(SUM(parval) / 1e9, 2) as total_value_at_risk_B,
        ROUND(AVG(parval) / 1e6, 2) as avg_parcel_value_M,
        COUNT(DISTINCT scity) as cities_affected
    FROM ca_parcels
    WHERE fema_flood_zone IN ('V', 'VE')
        AND county IN ('los-angeles', 'orange', 'san-diego')
""").show(truncate=False)

## 6. Spatial Join: Flood Zones × Overture Building Footprints

Cross-reference flood-risk parcels with the **Overture Maps Foundation** building footprints dataset to estimate the number of actual structures exposed to flooding. This adds a physical-infrastructure lens on top of the parcel-based analysis.

In [8]:
# Load Overture buildings within California
import wkls

print("🏗️  Loading Overture Building Footprints for California...")
ca_buildings = sedona.sql(f"""
    SELECT id, geometry, height, num_floors, class, subtype
    FROM wherobots_open_data.overture_maps_foundation.buildings_building
    WHERE ST_Intersects(geometry, ST_GeomFromWKT('{wkls.us['ca'].wkt()}'))
""")
ca_buildings.createOrReplaceTempView("ca_buildings")
print(f"   Buildings loaded: {ca_buildings.count():,}")

# Spatial join — buildings sitting within high-risk flood zone parcels
print("\n🔗 Spatial join: buildings × high-risk flood zone parcels...")
flood_buildings = sedona.sql("""
    SELECT
        p.county,
        p.scity as city_name,
        p.fema_flood_zone,
        b.class as building_class,
        b.num_floors,
        b.height,
        p.parval,
        p.yearbuilt
    FROM ca_parcels p
    JOIN ca_buildings b
        ON ST_Intersects(b.geometry, p.geometry)
    WHERE p.fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'V', 'VE', 'A99')
        AND p.county IN ('los-angeles', 'orange', 'san-diego')
""")
flood_buildings.createOrReplaceTempView("flood_buildings")
total_flood_bldgs = flood_buildings.count()
print(f"   Buildings in flood zones: {total_flood_bldgs:,}")

# Summary by county & flood zone
print("\n📊 Buildings at Risk by County and Flood Zone Type:")
sedona.sql("""
    SELECT
        county,
        fema_flood_zone,
        COUNT(*) as buildings_at_risk,
        COUNT(DISTINCT city_name) as cities_affected,
        ROUND(AVG(parval) / 1e6, 2) as avg_parcel_value_M,
        ROUND(AVG(num_floors), 1) as avg_floors,
        ROUND(AVG(height), 1) as avg_height_m
    FROM flood_buildings
    GROUP BY county, fema_flood_zone
    ORDER BY county, buildings_at_risk DESC
""").show(30, truncate=False)

## 7. Financial Exposure: Property Value at Risk

Quantify the **dollar value of real estate** sitting in FEMA-designated flood zones — broken down by city, zone severity, and property characteristics.

In [9]:
# Financial exposure — total value at risk by city
print("💰 Financial Exposure: Top 15 Cities by Total Property Value at Risk")
print("=" * 100)

financial_exposure = sedona.sql("""
    SELECT
        scity as city_name,
        county,
        COUNT(*) as parcels_at_risk,
        ROUND(SUM(parval) / 1e9, 2) as total_value_at_risk_B,
        ROUND(AVG(parval) / 1e6, 2) as avg_parcel_value_M,
        ROUND(MAX(parval) / 1e6, 2) as max_parcel_value_M,
        SUM(CASE WHEN fema_flood_zone IN ('V', 'VE') THEN 1 ELSE 0 END) as v_zone_parcels,
        ROUND(SUM(CASE WHEN fema_flood_zone IN ('V', 'VE') THEN parval ELSE 0 END) / 1e9, 2) as v_zone_value_B,
        ROUND(AVG(yearbuilt), 0) as avg_year_built,
        ROUND(SUM(CASE WHEN yearbuilt < 1975 THEN parval ELSE 0 END) / 1e9, 2) as pre_1975_value_B
    FROM ca_parcels
    WHERE fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'V', 'VE', 'A99')
        AND county IN ('los-angeles', 'orange', 'san-diego')
        AND scity IS NOT NULL AND scity != ''
        AND parval > 0
    GROUP BY scity, county
    ORDER BY total_value_at_risk_B DESC
    LIMIT 15
""")
financial_exposure.show(15, truncate=False)

# Grand totals
print("\n📊 Grand Total — All Coastal Counties:")
sedona.sql("""
    SELECT
        COUNT(*) as total_parcels_at_risk,
        ROUND(SUM(parval) / 1e9, 2) as total_value_at_risk_B,
        ROUND(AVG(parval) / 1e6, 2) as avg_parcel_value_M,
        SUM(CASE WHEN fema_flood_zone IN ('V', 'VE') THEN 1 ELSE 0 END) as v_zone_parcels,
        ROUND(SUM(CASE WHEN fema_flood_zone IN ('V', 'VE') THEN parval ELSE 0 END) / 1e9, 2) as v_zone_value_B,
        ROUND(
            SUM(CASE WHEN fema_flood_zone IN ('V', 'VE') THEN parval ELSE 0 END) * 100.0 /
            NULLIF(SUM(parval), 0),
        1) as v_zone_pct_of_total
    FROM ca_parcels
    WHERE fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'V', 'VE', 'A99')
        AND county IN ('los-angeles', 'orange', 'san-diego')
        AND parval > 0
""").show(truncate=False)

# Value distribution by risk tier
print("\n🎯 Value at Risk by Risk Tier:")
sedona.sql("""
    SELECT
        CASE
            WHEN fema_flood_zone IN ('V', 'VE') THEN '🔴 Coastal High Hazard (V/VE)'
            WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH') THEN '🟠 High Risk (A Zones)'
            WHEN fema_flood_zone = 'A99' THEN '🟡 Future Risk (A99)'
            ELSE '⚪ Other'
        END as risk_tier,
        COUNT(*) as parcels,
        ROUND(SUM(parval) / 1e9, 2) as total_value_B,
        ROUND(AVG(parval) / 1e6, 2) as avg_value_M
    FROM ca_parcels
    WHERE fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'V', 'VE', 'A99')
        AND county IN ('los-angeles', 'orange', 'san-diego')
        AND parval > 0
    GROUP BY
        CASE
            WHEN fema_flood_zone IN ('V', 'VE') THEN '🔴 Coastal High Hazard (V/VE)'
            WHEN fema_flood_zone IN ('A', 'AE', 'AO', 'AH') THEN '🟠 High Risk (A Zones)'
            WHEN fema_flood_zone = 'A99' THEN '🟡 Future Risk (A99)'
            ELSE '⚪ Other'
        END
    ORDER BY total_value_B DESC
""").show(truncate=False)

## 8. Weather Events Correlation

Overlay **historical weather events** (rain, storm, precipitation) from the Wherobots Pro weather dataset onto the flood-risk zones to see which areas have the highest compound exposure: flood-designated *and* frequently hit by severe weather.

In [10]:
# Load California weather events
print("🌧️  Loading California Weather Events...")
ca_weather = sedona.sql("""
    SELECT Type, Severity, City, County, `StartTime(UTC)`, `EndTime(UTC)`,
           `Precipitation(in)` as Precipitation, geometry
    FROM wherobots_pro_data.weather.weather_events
    WHERE State = 'CA'
        AND Type IN ('Rain', 'Storm', 'Precipitation', 'Cold')
""")
ca_weather.createOrReplaceTempView("ca_weather")
print(f"   Weather events loaded: {ca_weather.count():,}")

# Weather event summary by type and severity
print("\n📊 Weather Events in California by Type and Severity:")
sedona.sql("""
    SELECT
        Type,
        Severity,
        COUNT(*) as event_count,
        ROUND(AVG(CAST(Precipitation AS DOUBLE)), 2) as avg_precipitation
    FROM ca_weather
    GROUP BY Type, Severity
    ORDER BY event_count DESC
""").show(20, truncate=False)

# Spatial join — weather events overlapping flood-risk parcels
print("\n🔗 Spatial Join: Weather Events × Flood-Risk Parcels...")
weather_flood_join = sedona.sql("""
    SELECT
        w.Type as weather_type,
        w.Severity as weather_severity,
        p.county,
        p.scity as city_name,
        p.fema_flood_zone,
        CAST(w.Precipitation AS DOUBLE) as precipitation,
        p.parval
    FROM ca_parcels p
    JOIN ca_weather w
        ON ST_Intersects(w.geometry, p.geometry)
    WHERE p.fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'V', 'VE', 'A99')
        AND p.county IN ('los-angeles', 'orange', 'san-diego')
""")
weather_flood_join.createOrReplaceTempView("weather_flood_join")

# Compound risk: cities with both flood designation AND frequent severe weather
print("\n⚠️  Compound Risk: Flood Zone Parcels with Severe Weather Exposure (by City)")
sedona.sql("""
    SELECT
        city_name,
        county,
        weather_type,
        COUNT(*) as weather_flood_overlaps,
        COUNT(DISTINCT fema_flood_zone) as flood_zone_types,
        ROUND(AVG(precipitation), 2) as avg_precipitation,
        ROUND(SUM(parval) / 1e9, 2) as total_parcel_value_B
    FROM weather_flood_join
    WHERE city_name IS NOT NULL AND city_name != ''
    GROUP BY city_name, county, weather_type
    ORDER BY weather_flood_overlaps DESC
    LIMIT 20
""").show(20, truncate=False)

## 9. Spatial Visualization: Mapping Flood Risk

Generate interactive maps using **SedonaKepler** to visualize the geographic distribution of flood-risk parcels across California's coastal counties.

In [ ]:
# Visualization: Map of flood-risk parcels across coastal counties
from sedona.maps.SedonaKepler import SedonaKepler
from IPython.display import display, HTML

# --- Map 1: V-Zone (highest risk) parcels ---
print("🗺️  Map 1: Coastal V-Zone Parcels (Highest Flood Risk)")
v_zone_map_df = sedona.sql("""
    SELECT
        geometry,
        scity as city,
        county,
        fema_flood_zone,
        COALESCE(parval, 0) as parval,
        COALESCE(yearbuilt, 0) as yearbuilt
    FROM ca_parcels
    WHERE fema_flood_zone IN ('V', 'VE')
        AND county IN ('los-angeles', 'orange', 'san-diego')
        AND geometry IS NOT NULL
    LIMIT 5000
""")

map_vzone = SedonaKepler.create_map(df=v_zone_map_df, name="V-Zone Parcels")

# _repr_html_() returns bytes in this version of KeplerGL — decode and wrap in iframe
html_bytes = map_vzone._repr_html_()
html_str = html_bytes.decode('utf-8') if isinstance(html_bytes, bytes) else html_bytes
display(HTML(html_str))

## 10. Summary & Key Findings

Pull together the headline numbers from the entire analysis into a single consolidated view.

In [ ]:
# ===== EXECUTIVE SUMMARY =====
print("=" * 100)
print("🏖️  CALIFORNIA COASTAL FLOOD RISK ANALYSIS — EXECUTIVE SUMMARY")
print("=" * 100)

# 1. Total parcels at risk
summary = sedona.sql("""
    SELECT
        COUNT(*) as total_parcels_at_risk,
        ROUND(SUM(parval) / 1e9, 2) as total_value_at_risk_B,
        COUNT(DISTINCT scity) as cities_affected,
        SUM(CASE WHEN fema_flood_zone IN ('V', 'VE') THEN 1 ELSE 0 END) as coastal_v_zone_parcels,
        ROUND(SUM(CASE WHEN fema_flood_zone IN ('V', 'VE') THEN parval ELSE 0 END) / 1e9, 2) as v_zone_value_B
    FROM ca_parcels
    WHERE fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'V', 'VE', 'A99')
        AND county IN ('los-angeles', 'orange', 'san-diego')
        AND parval > 0
""").collect()[0]

print(f"\n📊 HEADLINE NUMBERS")
print(f"   Total parcels in FEMA flood zones:   {summary['total_parcels_at_risk']:,}")
print(f"   Total property value at risk:        ${summary['total_value_at_risk_B']}B")
print(f"   Cities affected:                     {summary['cities_affected']}")
print(f"   Coastal V-Zone (highest risk) parcels: {summary['coastal_v_zone_parcels']:,}")
print(f"   V-Zone property value at risk:       ${summary['v_zone_value_B']}B")

# 2. County breakdown
print(f"\n📍 BY COUNTY:")
sedona.sql("""
    SELECT
        county,
        COUNT(*) as parcels_at_risk,
        ROUND(SUM(parval) / 1e9, 2) as value_at_risk_B,
        SUM(CASE WHEN fema_flood_zone IN ('V', 'VE') THEN 1 ELSE 0 END) as v_zone_parcels
    FROM ca_parcels
    WHERE fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'V', 'VE', 'A99')
        AND county IN ('los-angeles', 'orange', 'san-diego')
        AND parval > 0
    GROUP BY county
    ORDER BY value_at_risk_B DESC
""").show(truncate=False)

# 3. Top 10 most-exposed cities
print("🏙️  TOP 10 MOST-EXPOSED CITIES (by total value at risk):")
sedona.sql("""
    SELECT
        scity as city,
        county,
        COUNT(*) as parcels,
        ROUND(SUM(parval) / 1e9, 2) as value_at_risk_B,
        SUM(CASE WHEN fema_flood_zone IN ('V', 'VE') THEN 1 ELSE 0 END) as v_zone
    FROM ca_parcels
    WHERE fema_flood_zone IN ('A', 'AE', 'AO', 'AH', 'V', 'VE', 'A99')
        AND county IN ('los-angeles', 'orange', 'san-diego')
        AND scity IS NOT NULL AND scity != ''
        AND parval > 0
    GROUP BY scity, county
    ORDER BY value_at_risk_B DESC
    LIMIT 10
""").show(truncate=False)

print("\n" + "=" * 100)
print("📋 METHODOLOGY NOTES:")
print("   • Data source: Regrid parcel data (sample) for 5 CA counties")
print("   • Coastal counties analyzed: Los Angeles, Orange, San Diego")
print("   • Flood designation: FEMA flood zone codes (A, AE, AO, AH, V, VE, A99)")
print("   • Property values: Assessed parcel values from county records")
print("   • Building data: Overture Maps Foundation building footprints")
print("   • Weather overlay: Wherobots Pro weather events (Rain, Storm, Precipitation)")
print("   • Spatial operations: Apache Sedona ST_Intersects for all joins")
print("   • Validated against: First Street Foundation data for San Diego ✅")
print("=" * 100)